## Business Objective

#### Segmenting customers into behavioral segments using RFM to identify the highest-value customers and those at risk of churn, with the aim of improving campaign targeting, increasing revenue from existing customers, and reducing customer attrition.

## Obtaining Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
UCI_df = pd.read_csv(r"C:\Users\FuTuRe\Desktop\online_retail_II.csv", sep=";", encoding="latin1")
UCI_df.head(5)

,invoice,stockcode,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,01/12/2009 07:45,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,01/12/2009 07:45,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,01/12/2009 07:45,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,01/12/2009 07:45,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,01/12/2009 07:45,1.25,13085.0,United Kingdom


In [3]:
UCI_df.shape

(1048575, 8)

## Scrubbing Data

In [4]:
UCI_df[UCI_df.duplicated(keep=False)]

,invoice,stockcode,description,quantity,invoice_date,price,customer_id,country
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,01/12/2009 11:34,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,01/12/2009 11:34,3.75,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,01/12/2009 11:34,3.75,16329.0,United Kingdom
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,01/12/2009 11:34,0.65,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,01/12/2009 11:34,0.85,16329.0,United Kingdom
...,...,...,...,...,...,...,...,...
1048349,580469,22079,RIBBON REEL HEARTS DESIGN,2,04/12/2011 12:32,1.65,14583.0,United Kingdom
1048506,580501,23101,SILVER STARS TABLE DECORATION,2,04/12/2011 13:00,0.83,14546.0,United Kingdom
1048509,580501,23438,RED SPOT GIFT BAG LARGE,2,04/12/2011 13:00,1.25,14546.0,United Kingdom
1048510,580501,23438,RED SPOT GIFT BAG LARGE,2,04/12/2011 13:00,1.25,14546.0,United Kingdom


In [5]:
UCI_df.duplicated().sum()

np.int64(34150)

In [6]:
UCI_df.drop_duplicates(keep = 'first', inplace= True)

In [7]:
UCI_df.shape

(1014425, 8)

In [8]:
UCI_df.isna().sum()

invoice              0
stockcode            0
description       4265
quantity             0
invoice_date         0
price                0
customer_id     228826
country              0
dtype: int64

In [9]:
UCI_df.dropna(subset=["customer_id"], inplace= True)

In [10]:
UCI_df.isna().sum()

invoice         0
stockcode       0
description     0
quantity        0
invoice_date    0
price           0
customer_id     0
country         0
dtype: int64

In [11]:
UCI_df["invoice_date"] = pd.to_datetime(UCI_df["invoice_date"], format = "%d/%m/%Y %H:%M").dt.date

In [12]:
UCI_df["invoice_date"] = pd.to_datetime(UCI_df["invoice_date"])

In [13]:
UCI_df["customer_id"] = UCI_df["customer_id"].astype("int64")

In [14]:
UCI_df.dtypes

invoice                   str
stockcode                 str
description               str
quantity                int64
invoice_date    datetime64[s]
price                 float64
customer_id             int64
country                   str
dtype: object

In [15]:
UCI_df.shape

(785599, 8)

In [16]:
UCI_df["invoice"] = UCI_df["invoice"].astype(str)
UCI_df = UCI_df[~UCI_df["invoice"].str.startswith("C")].copy()

In [17]:
UCI_df.shape

(767439, 8)

## Exploring Data

In [18]:
UCI_df[["quantity", "price"]].describe()

,quantity,price
count,767439.000000,767439.000000
mean,13.417992,3.226502
std,115.063721,29.846881
min,1.000000,0.000000
25%,2.000000,1.250000
50%,6.000000,1.950000
75%,12.000000,3.750000
max,74215.000000,10953.500000


In [19]:
print(UCI_df["quantity"].sort_values(ascending=False))
print("*" * 50)
print(UCI_df["price"].sort_values(ascending=False))

587080     74215
90857      19152
127166     12960
127168     12960
127169     12744
           ...  
87060          1
1048548        1
1048531        1
1048557        1
1048569        1
Name: quantity, Length: 767439, dtype: int64
**************************************************
135013    10953.50
358639    10468.80
74356      8985.60
698843     8142.75
129987     6958.17
            ...   
811118        0.00
884131        0.00
471775        0.00
471776        0.00
962058        0.00
Name: price, Length: 767439, dtype: float64


In [20]:
print(UCI_df["quantity"].quantile([0.95, 0.99, 0.995, 0.999]))
print("*" * 50)
print(UCI_df["price"].quantile([0.95, 0.99, 0.995, 0.999]))

0.950     36.0
0.990    144.0
0.995    216.0
0.999    576.0
Name: quantity, dtype: float64
**************************************************
0.950     8.50
0.990    14.95
0.995    18.00
0.999    49.95
Name: price, dtype: float64


In [21]:
UCI_df = UCI_df[(UCI_df["quantity"] > 0) & (UCI_df["price"] > 0)]

In [22]:
q_hi = UCI_df["quantity"].quantile(0.995)
p_hi = UCI_df["price"].quantile(0.999)

UCI_df["quantity"] = UCI_df["quantity"].clip(upper=q_hi)
UCI_df["price"] = UCI_df["price"].clip(upper=p_hi)


In [23]:
print("Qty capped rows:", (UCI_df["quantity"] == q_hi).sum())
print("Price capped rows:", (UCI_df["price"] == p_hi).sum())

Qty capped rows: 3841
Price capped rows: 802


In [24]:
UCI_df[["quantity", "price"]].describe()

,quantity,price
count,767369.000000,767369.000000
mean,11.647704,2.977443
std,23.428302,3.447511
min,1.000000,0.001000
25%,2.000000,1.250000
50%,6.000000,1.950000
75%,12.000000,3.750000
max,216.000000,49.950000


In [25]:
UCI_df.head()

,invoice,stockcode,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085,United Kingdom


In [26]:
UCI_df.isna().sum()

invoice         0
stockcode       0
description     0
quantity        0
invoice_date    0
price           0
customer_id     0
country         0
dtype: int64

In [27]:
import datetime as dt
df = UCI_df.copy()


df["TotalPrice"] = df["quantity"] * df["price"]

# Snapshot date
snapshot_date = df["invoice_date"].max() + dt.timedelta(days=1)

# RFM
rfm = df.groupby("customer_id").agg(
    Recency=("invoice_date", lambda x: (snapshot_date - x.max()).days),
    Frequency=("invoice", "nunique"),
    Monetary=("TotalPrice", "sum")
)

# Scores
rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)

rfm["RFM_Score"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)

def segment(row):
    R, F, M = row["R_Score"], row["F_Score"], row["M_Score"]
    if R >= 4 and F >= 4 and M >= 4:
        return "Champions"
    if R <= 2 and F >= 4 and M >= 4:
        return "Can't lose"
    if R <= 2 and (F >= 3 or M >= 3):
        return "At risk"
    if R == 5 and F <= 2:
        return "New"
    if R >= 4 and F <= 3:
        return "Promising"
    if R == 1 and F <= 2 and M <= 2:
        return "Lost"
    return "At risk" if R <= 2 else "Promising"

rfm["Segment"] = rfm.apply(segment, axis=1)

In [28]:
rfm["Segment"].value_counts(normalize=True)

Segment
Promising     0.351536
At risk       0.230375
Champions     0.218089
Lost          0.129352
Can't lose    0.040102
New           0.030546
Name: proportion, dtype: float64

In [29]:
df.shape

(767369, 9)

In [30]:
rfm

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
customer_id,,,,,,,,
12346,321,12,597.50,2,5,3,253,At risk
12347,35,7,4671.75,4,4,5,445,Champions
12348,71,5,2019.40,3,4,4,344,Promising
12349,14,4,3828.54,5,3,5,535,Promising
12350,306,1,334.40,2,1,2,212,At risk
...,...,...,...,...,...,...,...,...
18283,5,21,2456.90,5,5,4,554,Champions
18284,427,1,461.63,1,2,2,122,Lost
18285,656,1,426.95,1,2,2,122,Lost


In [31]:
rfm.groupby("Segment")[["Recency","Frequency","Monetary"]].median().sort_values("Monetary", ascending=False)

,Recency,Frequency,Monetary
Segment,,,
Champions,18.0,11.0,3907.825
Can't lose,325.0,7.0,2162.930
Promising,67.0,3.0,807.510
At risk,372.0,2.0,534.175
New,12.0,2.0,371.200
Lost,560.0,1.0,208.310


In [34]:
clean_uci = df.to_csv(r"C:\Users\FuTuRe\Desktop\UCI.csv", index=False)
rfm_cus = rfm.to_csv(r"C:\Users\FuTuRe\Desktop\customer_rfm.csv", index=True)